Author: Enikő Gergely <br>
Date: October 2026
    
# Automated Silhouette and Angle calculation for Hitter Posture Analysis

This notebook includes a process to draw silhouette and a Back-Glutel line on the cropped hitter images to collect angle data for later hitter posture analysis.
Angle data is saved to an excel file including game date information.

The goal is to automate the data collection process for my previous  [study](https://github.com/enikogergely26/YelichPostureAnalysis/blob/main/Christian%20Yelich%20Posture%20Analysis%20-%20EnikoGergely.pdf) and see if the automated process gives the same results that I got with the manual data collection. If results are the same, the automated process can be used to analyze other players and analyze further the relationship between the Back-Gluteal angle and the batting performance. 

# Import Packages

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import requests
import os
import pandas as pd

# Functions:

In [2]:

# Function to get game date using game_pk:
def get_date_from_game_pk(game_pk:int):

    """
    Gets the game date from the MLB Stats API using the game ID.

    Args:
        game_pk (int): The unique ID of the MLB game.

    Returns:
        str: The game date in YYYY-MM-DD format.
    """
    
    url = f"https://statsapi.mlb.com/api/v1/schedule?gamePk={game_pk}"
    data = requests.get(url).json()

    game_date = data["dates"][0]["games"][0]["gameDate"]

    return game_date[:10]


def get_batting_hand(player_id: int):
    """
    Gets a player's batting handedness from the MLB Stats API.

    Args:
        player_id (int): The MLB player ID.

    Returns:
        str: 'L' for left-handed, 'R' for right-handed,
             or 'S' for a switch hitter.
    """
    url = f"https://statsapi.mlb.com/api/v1/people/{player_id}"
    data = requests.get(url).json()

    return data["people"][0]["batSide"]["code"]


# Function to get back-gluteal line and its angle:
def back_gluteal_line(image_path: Path, output_dir: Path, handedness: str):
    """
        Detects the batter's silhouette, identifies the Back-Gluteal line, calculates its angle,
        and saves the annotated image to the selected output folder.

        Args:
            image_path (Path): Path to the input image of the batter.
            output_dir(Path): Directory where the annotated image will be saved.
            handedness (str): Batter's handedness ('R' for righ-handed, 'L' for 
            left-handed). If 'R', the image is flipped horizontally before processing.

        Returns:
            dict: A dictionary containing the input file name, calculated Back-Gluteal angle in degrees,
                and game date.
            None: If the image cannot be loaded or a silhouette or contour cannot be found.
    """

    # --------------------------------------------------
    # Load image
    # --------------------------------------------------

    # get game date:
    image_name = Path(image_path).name
    game_date = get_date_from_game_pk(image_name[14:20])
    print ("Game date: ", game_date)

    # load image:
    image = cv2.imread(str(image_path))

    if image is None:
        print(f"[ERROR] Cannot read: {image_path}")
        return None

    # if 'R', flip horizontally:
    if handedness == 'R':
        image = cv2.flip(image, 1)

    # image shape:
    print(f"Image shape: {image.shape}")

    h, w, _ = image.shape

    # --------------------------------------------------
    # YOLO segmentation and find best mask
    # --------------------------------------------------
    
    model = YOLO("yolov8n-seg.pt")
    
    results = model(image)[0]

    if results.masks is None:
        print("[ERROR] No masks detected")
        return None

    # get 40% of the width and half of height - this point should be part of the silhouette
    image_center = np.array([int(w * 0.4), int(h / 2)])
    print("Selected point to be part of the silhouette:", image_center)

    # select mask which includes the selected point and has the largest condifence:
    best_mask = None
    best_confidence = 0

    for mask, box in zip(results.masks.data, results.boxes):

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        # Only person masks
        if class_id != 0:
            continue

        print(f"Person detected | confidence={confidence:.2f}")

        # --------------------------------------------------
        # Resize mask to the original image size
        # --------------------------------------------------

        mask_np = mask.cpu().numpy()

        mask_resized = cv2.resize(
            mask_np,
            (w, h),
            interpolation=cv2.INTER_NEAREST
        )

        # binary mask:
        mask_binary = mask_resized > 0.5

        # --------------------------------------------------
        # Is the selected point in the mask?
        # --------------------------------------------------

        x = image_center[0]
        y = image_center[1]

        point_inside = mask_binary[y, x]

        print(f"Point inside mask: {point_inside}")

        # If not, drop this mask:
        if not point_inside:
            continue

        # --------------------------------------------------
        # If point is included, select the mask with the largest confidence.
        # --------------------------------------------------

        if confidence > best_confidence:

            best_confidence = confidence
            best_mask = mask_binary


    # --------------------------------------------------
    # Check
    # --------------------------------------------------

    if best_mask is None:

        print("[ERROR] No person mask contains the selected point")
        return None

    print(
        f"Selected person | confidence={best_confidence:.2f}"
    )

    # --------------------------------------------------
    # Find contour
    # --------------------------------------------------

    mask_binary = best_mask.astype(np.uint8) * 255

    kernel = np.ones((15, 15), np.uint8)

    # connect all countour parts to have only one contour:
    mask_binary = cv2.morphologyEx(
        mask_binary.astype(np.uint8),
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        mask_binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        print("[ERROR] No contour found")
        return None

    # select the contour with the largest area for player contour
    player_contour = max(
        contours,
        key=cv2.contourArea
    )

    print(
        "Selected silhouette area:",
        cv2.contourArea(player_contour)
    )


    # --------------------------------------------------
    # Draw silhouette
    # --------------------------------------------------

    cv2.drawContours(
        image,
        [player_contour],
        -1,
        (0, 255, 0),
        3
    )

    plt.figure(figsize=(12, 8))

    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    plt.axis("off")
    plt.title("Detected batter silhouette")
    plt.show()

    # --------------------------------------------------
    # Largest countour
    # --------------------------------------------------
    
    # select the contour with the largest area
    largest_contour = 0
    largest_area = 0

    for i, contour in enumerate(contours):

        area = cv2.contourArea(contour)

        x, y, w_c, h_c = cv2.boundingRect(contour)

        print(
            f"Contour {i}: "
            f"area={area:.0f}, "
            f"x={x}, y={y}, "
            f"w={w_c}, h={h_c}"
        )

        if area > largest_area:
            largest_area = area
            largest_contour = i

    print ("Selected contour = ", i)


    # --------------------------------------------------
    # Find Back point
    # --------------------------------------------------

    # get the top 5-30% of heigth of the image
    y_Back_bottom = int(h *0.3 )
    print("Selected point:", y_Back_bottom)

    y_Back_top = int(h *0.05 )
    print("Selected point:", y_Back_top)


    # Filter contour pairs where y_Back_bottom > y > y_Back_top 
    contour = contours[largest_contour].reshape(-1, 2)
    filtered = [[[x, y]] for x, y in contour if y < y_Back_bottom  and y > y_Back_top]
    filtered_array = np.array(filtered)

    # find the smallest x's index:
    min_x_idx = np.argmin(filtered_array[:, 0, 0])

    # the index's x and y:
    back_point_x = filtered_array[min_x_idx, 0, 0]
    back_point_y = filtered_array[min_x_idx, 0, 1]

    print(f"Back point X: {back_point_x}, and its Y: {back_point_y}")

    # --------------------------------------------------
    # Find Gluteal point
    # --------------------------------------------------

    # get the top 30-60% of heigth of the image
    y_Gluteal_bottom = int(h *0.6 )
    print("Selected point:", y_Gluteal_bottom)

    # Filter pairs where y_Gluteal_bottom > y > y_Back_bottom
    filtered = [[[x, y]] for x, y in contour if y > y_Back_bottom and y < y_Gluteal_bottom]
    filtered_array = np.array(filtered)

    # find the smallest x's index:
    min_x_idx = np.argmin(filtered_array[:, 0, 0])

    # the index's x and y:
    gluteal_point_x = filtered_array[min_x_idx, 0, 0]
    gluteal_point_y = filtered_array[min_x_idx, 0, 1]

    print(f"Gluteal point X: {gluteal_point_x}, and its Y: {gluteal_point_y}")

    #filtered_player_contour = filtered_array.reshape(-1, 1, 2)

    # --------------------------------------------------
    # Draw the Back-Gluteal Line and horizontal segmentation lines
    # --------------------------------------------------

    P1 = np.array([back_point_x, back_point_y])  # back
    P2 = np.array([gluteal_point_x, gluteal_point_y])  # glutes


    # Draw a line on the two points
    dx = P2[0] - P1[0]
    dy = P2[1] - P1[1]

    length = np.sqrt(dx**2 + dy**2)

    ux = dx / length
    uy = dy / length

    # extension of the line
    extension = 50

    A = P1 - extension * np.array([ux, uy])
    B = P2 + extension * np.array([ux, uy])


    cv2.line(
        image,
        tuple(A.astype(int)),
        tuple(B.astype(int)),
        (0, 0, 255),
        3
    )

    # add horizontal lines for the segmentations:
    # Bottom of the Gluteal section
    cv2.line(
        image,
        (0, y_Gluteal_bottom),
        (w, y_Gluteal_bottom),
        (255, 0, 0),
        2
    )
    # Bottom of the Back section/ Top of the Gluteal section
    cv2.line(
        image,
        (0, y_Back_bottom),
        (w, y_Back_bottom),
        (255, 0, 0),
        2
    )
    # Top of the Back section
    cv2.line(
        image,
        (0, y_Back_top),
        (w, y_Back_top),
        (255, 0, 0),
        2
    )
    # --------------------------------------------------
    # Calculate the angle in degrees
    # --------------------------------------------------

    angle = np.degrees(np.arctan2(dy, dx))

    print(f"Angle: {angle:.2f}°")
    
    # --------------------------------------------------
    # Display the image
    # --------------------------------------------------

   # if 'R', flip back horizontally:
    if handedness == 'R':
        image = cv2.flip(image, 1)

    plt.figure(figsize=(12, 8))

    plt.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    plt.axis("off")
    plt.title("Detected body contour")
    plt.show()


    # --------------------------------------------------
    # Save final image
    # --------------------------------------------------

    output_dir.mkdir(parents=True, exist_ok=True)

    out_path = output_dir / f"{game_date}_{image_name}"

    cv2.imwrite(str(out_path), image)
    print(f"[OK] -> {out_path}")

    # --------------------------------------------------
    # Return angle data
    # --------------------------------------------------
    
    return {"file_name": image_path.name, "angle": angle, "Date": game_date}

# Run Analysis:

## Inputs:
player_name (str): Player Name (as in folder name). <br>
player (int): A player ID (i.e. 12345).<br>
year (int): Year to be analyzed.<br>

In [3]:
player_name = 'Yelich'
player = 592885
year = 2024

### Handedness

In [4]:
handedness = get_batting_hand(player)

print(f"Batter handedness: {handedness}")

Batter handedness: L


### Analyze Images:

In [ ]:
# create empty list for results:
all_results = []

# folder where cropped images are saved
cropped_images_input_dir = Path(f"{player_name}({player})\cropped_images_{player_name}_{year}")

# loop through all the files and calculate angle
for file_name in os.listdir(cropped_images_input_dir):
    print(file_name)
    
    result = back_gluteal_line(cropped_images_input_dir /file_name, 
                               Path(f"{player_name}({player})/angle_results_{player_name}_{year}"), 
                               handedness)
    if result:
        all_results.append(result)

# Create dataframe:
all_results_df = pd.DataFrame(all_results) 


### Save results:

In [8]:
# Mean angle:
mean_angle = all_results_df['angle'].mean()
print(f"Mean angle for {year} is {round(mean_angle,2)}") 

# Save to excel
all_results_df.to_excel( Path(f"{player_name}({player})")/f"Angle_data_{player_name}_{year}.xlsx" , index=False)
print("Excel file has been saved to player folder.")

Mean angle for 2024 is 98.47
Excel file has been saved to player folder.
